In [ ]:
# relevant environment preparation
# failed!!!!!!
import hashlib
import os
import tarfile
import zipfile
import requests

#@save
DATA_HUB = dict()
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/'

def download(name, cache_dir=os.path.join('..', 'data')):  #@save
    """下载一个DATA_HUB中的文件，返回本地文件名"""
    assert name in DATA_HUB, f"{name} 不存在于 {DATA_HUB}"
    url, sha1_hash = DATA_HUB[name]
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, url.split('/')[-1])
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname  # 命中缓存
    print(f'正在从{url}下载{fname}...')
    r = requests.get(url, stream=True, verify=True)
    with open(fname, 'wb') as f:
        f.write(r.content)
    return fname

def download_extract(name, folder=None):  #@save
    """下载并解压zip/tar文件"""
    fname = download(name)
    base_dir = os.path.dirname(fname)
    data_dir, ext = os.path.splitext(fname)
    if ext == '.zip':
        fp = zipfile.ZipFile(fname, 'r')
    elif ext in ('.tar', '.gz'):
        fp = tarfile.open(fname, 'r')
    else:
        assert False, '只有zip/tar文件可以被解压缩'
    fp.extractall(base_dir)
    return os.path.join(base_dir, folder) if folder else data_dir

def download_all():  #@save
    """下载DATA_HUB中的所有文件"""
    for name in DATA_HUB:
        download(name)

In [ ]:
# accessing and reading data with pandas

# ! pip install --force-reinstall matplotlib==3.5.1 numpy==1.21.5
import numpy as np
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l

DATA_HUB['kaggle_house_train'] = (  #@save
    DATA_URL + 'kaggle_house_pred_train.csv',
    '585e9cc93e70b39160e7921475f9bcd7d31219ce')

DATA_HUB['kaggle_house_test'] = (  #@save
    DATA_URL + 'kaggle_house_pred_test.csv',
    'fa19780a7b011d9b009e8bff8e99922a8ee2eb90')

train_data = pd.read_csv(download('kaggle_house_train'))
test_data = pd.read_csv(download('kaggle_house_test'))

In [ ]:
# 检查数据
print(train_data.shape)
print(test_data.shape)

# 一共有80个特征，只提取看部分
print(train_data.iloc[0:4, [0, 1, 2, 3, -3, -2, -1]])
# 删除第一行的id因为不包含feature信息
all_features = pd.concat((train_data.iloc[:, 1:-1], test_data.iloc[:, 1:]))

# handling missing data imputation

## deletion

### listwise delstion
dataset is massive and missing portion is tiny  
### feauture dropping
the missing portion is high  

## simple imputation

for numerical and categorical datas  
### mean imputation 
normally distrubute  
### median imputation
robust! for skewed data  
### mode imputation
replace NaN with most common catagory  
### constant filling

## model-based imputation    
### KNN  
### Iterative Imputation (Regression): 
Treats the feature with missing values as a "target" and uses other features to predict it via linear regression or decision trees.
### Interpolation:
 Common in Time-Series data. It estimates missing values based on the points before and after in a sequence.
 ## missing indicator or categorical 'Unknown'


In [ ]:
# data preprocessing
# fill the missing part then normalize all data so that 
# all features are of equal importance

numeric_features = all_features.dtypes[all_features.dtypes != 'object'].index
all_features[numeric_features] = all_features[numeric_features].apply(
    lambda x: ((x - x.meanx())/(x.std()))
)
# fill the missing part with mean value like 0
all_features[numeric_features] = all_features[numeric_features].fiilna(0)

# one-hot coding for strings  
# benifits:learn better with more features; more convenient for SGD(numerical)
# Dummy_na=True will process missing information as valid features and 
# create an index feature for missing
all_features = pd.get_dummies(all_features,dummy_na=True)
all_features.shape
# see expansion 

# extract np from pd, then converting into tensor
n_train = train_data.shape[0]
train_features = torch.tensor(all_features[:n_train].values,dtype=torch.float32)
test_features = torch.tensor(all_features[n_train:].values,dtype=torch.float32)
# to match the shape of outputs
train_labels = torch.tensor(train_data.SalePrice.values.reshape(-1,1), dtype=torch.float32)

In [ ]:
loss = nn.MSELoss()
in_features = train_features.shape[1]
def net():
    net = nn.Sequential(nn.Linear(in_features=in_features,out_features=1))
    return net

# taking the log of prices to stabalize the vriance
# calculate root mean squared error
def log_rmse(net,features,labels):
    preds = net(features)
    clipped_preds = torch.clamp(features, 1,float('inf'))
    rmse = torch.sqrt(loss(torch.log(labels),torch.log(clipped_preds
    )))
    return rmse.item()

In [ ]:
def train(net,train_features,train_labels,test_features
          ,test_labels,num_epochs,batch_size,decay_rate,learning_rate):
    train_list = []
    test_list = []
    train_iter = d2l.load_array((train_features,train_labels),batch_size)
    optimizer = torch.optim.Adam(net.parameters(),lr=learning_rate,weight_decay=decay_rate)
    for epoch in range(num_epochs):
        for X,y in train_iter:
            optimizer.zero_grad()
            loss(net(X), y).backward()
            optimizer.step()
        train_list.append(log_rmse(net, train_features, train_labels))
        if test_labels is not None:
            test_list.append(log_rmse(net,test_features,test_labels))
    return train_list, test_list

In [ ]:
# k-fold-validation(rotate the exam)

# def get_k_fold_data(k,i,X,y):
#     assert k > 1
#     fold_size = X.shape[0] // k
#     X_train ,y_train = None, None
#     X_valid, y_valid = None, None
#     for j in range(k):
#         idx = slice(j*fold_size, (j+1)*fold_size)
#         X_part ,y_part = X[idx,:], y[idx]
#         if j == i:
#             X_valid, y_valid = X_part ,y_part
#         elif X_train is None:
#             X_train ,y_train = X_part, y_part
#         else:
#             X_train = torch.cat([X_train,X_part],0)
#             y_train = torch.cat([y_train,y_part],0)
#     return X_train,y_train,X_valid,y_valid
        
def get_k_fold_data(k, i, X, y):
    assert k > 1
    fold_size = X.shape[0] // k
    
    X_train_list, y_train_list = [], [] # Use lists instead of None
    X_valid, y_valid = None, None
    
    for j in range(k):
        idx = slice(j * fold_size, (j + 1) * fold_size)
        X_part, y_part = X[idx, :], y[idx]
        
        if j == i:
            X_valid, y_valid = X_part, y_part
        else:
            X_train_list.append(X_part)
            y_train_list.append(y_part)
            
    # Concatenate everything once at the very end
    X_train = torch.cat(X_train_list, 0)
    y_train = torch.cat(y_train_list, 0)
    
    return X_train, y_train, X_valid, y_valid


def k_fold(k,X_train,y_train,num_epochs,learning_rate,decay_rate,batch_size):
    X_train_sum, X_test_sum = 0, 0
    
    for i in range(k):
        X_t, y_t, X_v, y_v=get_k_fold_data(k,i,X_train,y_train)
        
        net = net()
        train_list, valid_list = train(net,X_t,y_t,X_v,y_v,num_epochs
                                           ,batch_size=batch_size,decay_rate=decay_rate,learning_rate=learning_rate)
        
        X_train_sum += train_list[-1]
        X_test_sum += valid_list[-1]
        
    return X_train_sum / k, X_test_sum / k

In [ ]:
# modify hyperparameters

k, num_epochs, lr, weight_decay, batch_size = 5, 100, 5, 0, 64
train_l, valid_l = k_fold(k, train_features, train_labels, num_epochs, lr,
                          weight_decay, batch_size)
print(f'{k}-折验证: 平均训练log rmse: {float(train_l):f}, '
      f'平均验证log rmse: {float(valid_l):f}')

In [ ]:
def train_and_pred(train_features, test_features, train_labels, test_data,
                   num_epochs, lr, weight_decay, batch_size):
    net = net()
    train_ls, _ = train(net, train_features, train_labels, None, None,
                        num_epochs, lr, weight_decay, batch_size)
    # no need to caculate gradient so use detach 
    # pd only accept numpy
    preds = net(test_features).detach().numpy()

    test_data['ScalePrice'] = pd.Series(preds.reshape(1,-1)[0])
    submission = pd.concat([test_data['id'],test_data["ScalePrice"]])
    submission.to_csv('submission.csv', index=False)
